In [1]:
!pip install libpysal spreg esda geopandas matplotlib

  Using cached libpysal-4.14.1-py3-none-any.whl.metadata (5.0 kB)
  Using cached spreg-1.8.5-py3-none-any.whl.metadata (1.7 kB)
  Using cached geopandas-1.1.3-py3-none-any.whl.metadata (2.3 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached shapely-2.1.2-cp311-cp311-macosx_10_9_x86_64.whl.metadata (6.8 kB)
  Using cached pyogrio-0.12.1-cp311-cp311-macosx_12_0_x86_64.whl.metadata (5.9 kB)
  Using cached pyproj-3.7.2-cp311-cp311-macosx_13_0_x86_64.whl.metadata (31 kB)
Using cached libpysal-4.14.1-py3-none-any.whl (2.5 MB)
Using cached spreg-1.8.5-py3-none-any.whl (396 kB)
Using cached geopandas-1.1.3-py3-none-any.whl (342 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached pyogrio-0.12.1-cp311-cp311-macosx_12_0_x86_64.whl (25.2 MB)
Using cached pyproj-3.7.2-cp311-cp311-macosx_13_0_x86_64.whl (6.2 MB)
Using cached shapely-2.1.2-cp311-cp311-macosx_10_9_x86_64.whl (1.8 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [esda

# SAR

In [1]:
"""
=============================================================
SAR (Spatial AutoRegressive) 모델 교육용 예제
libpysal + spreg 버전
=============================================================
설치 명령어:
    pip install libpysal spreg geopandas matplotlib

데이터셋 : Columbus 범죄 데이터 (libpysal 내장)
           - 49개 오하이오주 Columbus 행정구역
           - CRIME : 범죄율 (종속변수)
           - INC   : 가구 평균 소득
           - HOVAL : 주택 평균 가치

모델 : y = ρ·W·y + X·β + ε   (Spatial Lag / SAR)
추정 : MLE (Maximum Likelihood Estimation)
=============================================================
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import libpysal
from spreg import OLS, ML_Lag
from libpysal.weights import Queen

matplotlib.rcParams['axes.unicode_minus'] = False

In [2]:
# =============================================================
# STEP 1. 데이터 로딩
# =============================================================
print("=" * 60)
print("STEP 1. Columbus 데이터 로딩 (libpysal 내장)")
print("=" * 60)

# Columbus 내장 데이터 경로
dbf_path = libpysal.examples.get_path('columbus.dbf')
shp_path = libpysal.examples.get_path('columbus.shp')
gal_path = libpysal.examples.get_path('columbus.gal')

db = libpysal.io.open(dbf_path)

y = np.array(db.by_col('CRIME')).reshape(-1, 1)   # 종속변수
X = np.array([db.by_col('INC'),
              db.by_col('HOVAL')]).T               # 독립변수

n = len(y)
print(f"  지역 수      : {n}개")
print(f"  CRIME 평균   : {y.mean():.3f}")
print(f"  INC   평균   : {X[:,0].mean():.3f}")
print(f"  HOVAL 평균   : {X[:,1].mean():.3f}")

STEP 1. Columbus 데이터 로딩 (libpysal 내장)
  지역 수      : 49개
  CRIME 평균   : 35.129
  INC   평균   : 14.375
  HOVAL 평균   : 38.436


In [3]:
# =============================================================
# STEP 2. 공간 가중치 행렬 W 생성
# =============================================================
print("\n" + "=" * 60)
print("STEP 2. 공간 가중치 행렬 W 생성 (Queen Contiguity)")
print("=" * 60)

# .gal 파일에서 직접 읽기
w = libpysal.io.open(gal_path).read()
w.transform = 'r'   # Row-standardization

print(f"  W 행렬 크기      : {w.n} x {w.n}")
print(f"  평균 이웃 수     : {w.mean_neighbors:.2f}")
print(f"  변환 방식        : Row-standardized (행 합 = 1)")


# =============================================================
# STEP 3. 공간 자기상관 확인 (Moran's I)
# =============================================================
print("\n" + "=" * 60)
print("STEP 3. Moran's I — 공간 자기상관 검정")
print("=" * 60)

from libpysal.weights import lag_spatial
from esda.moran import Moran

mi = Moran(y.flatten(), w)
print(f"  Moran's I    : {mi.I:.4f}")
print(f"  기대값 E[I]  : {mi.EI:.4f}")
print(f"  p-value      : {mi.p_norm:.4f}")
if mi.p_norm < 0.05:
    print(f"  결론         : 유의미한 양의 공간 자기상관 존재 → SAR 적합 ✓")
else:
    print(f"  결론         : 공간 자기상관 불분명")


# =============================================================
# STEP 4. OLS 모델 추정 (비교 기준)
# =============================================================
print("\n" + "=" * 60)
print("STEP 4. OLS 모델 추정 (비교 기준)")
print("=" * 60)

ols = OLS(y, X,
          w=w,
          name_y='CRIME',
          name_x=['INC', 'HOVAL'],
          name_ds='Columbus',
          spat_diag=True)   # 공간 진단 통계 포함

print(ols.summary)


# =============================================================
# STEP 5. SAR 모델 추정 (ML_Lag = Spatial Lag Model)
# =============================================================
print("\n" + "=" * 60)
print("STEP 5. SAR 모델 MLE 추정 (ML_Lag)")
print("=" * 60)
print("  모델 : CRIME = ρ·W·CRIME + β0 + β1·INC + β2·HOVAL + ε\n")

sar = ML_Lag(y, X,
             w=w,
             name_y='CRIME',
             name_x=['INC', 'HOVAL'],
             name_ds='Columbus')

print(sar.summary)


# =============================================================
# STEP 6. OLS vs SAR 핵심 결과 비교
# =============================================================
print("\n" + "=" * 60)
print("STEP 6. OLS vs SAR 핵심 결과 비교")
print("=" * 60)

# 계수 추출
ols_betas = ols.betas.flatten()     # [const, INC, HOVAL]
sar_betas = sar.betas.flatten()     # [const, INC, HOVAL, rho]
rho_hat   = sar.rho

print(f"\n  {'파라미터':<12} {'OLS':>10} {'SAR':>10}  변화")
print(f"  {'-'*46}")
labels = ['const', 'INC', 'HOVAL']
for i, lbl in enumerate(labels):
    diff = sar_betas[i] - ols_betas[i]
    print(f"  {lbl:<12} {ols_betas[i]:>10.4f} {sar_betas[i]:>10.4f}  Δ={diff:+.4f}")
print(f"  {'rho (ρ)':<12} {'N/A':>10} {rho_hat:>10.4f}")
print(f"\n  OLS  R²          : {ols.r2:.4f}")
print(f"  SAR  pseudo R²   : {sar.pr2:.4f}")
print(f"\n  해석:")
print(f"  ρ = {rho_hat:.4f} > 0 → 이웃 지역 범죄율이 높을수록 해당 지역도 높음")
print(f"  SAR pseudo R²가 OLS R²보다 높음 → 공간 구조 반영 시 설명력 향상")


# =============================================================
# STEP 7. 시각화
# =============================================================
print("\n" + "=" * 60)
print("STEP 7. 시각화")
print("=" * 60)

y_flat    = y.flatten()
y_pred_sar = sar.predy.flatten()
y_pred_ols = ols.predy.flatten()
res_sar   = sar.u.flatten()
res_ols   = ols.u.flatten()

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("SAR Model — Columbus Crime Analysis (libpysal)",
             fontsize=14, fontweight='bold', y=0.98)

# (1) Actual vs Predicted (SAR)
ax = axes[0, 0]
ax.scatter(y_flat, y_pred_sar, alpha=0.7, color='steelblue',
           edgecolors='white', s=60, label='SAR predicted')
mn, mx = min(y_flat.min(), y_pred_sar.min()), max(y_flat.max(), y_pred_sar.max())
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
ax.set_xlabel("Actual CRIME")
ax.set_ylabel("SAR Predicted CRIME")
ax.set_title("Actual vs SAR Predicted")
ax.legend()
ax.grid(alpha=0.3)

# (2) OLS vs SAR 잔차 비교
ax = axes[0, 1]
ax.scatter(range(n), res_ols, alpha=0.6, color='tomato', s=40,
           label=f'OLS  (std={res_ols.std():.2f})')
ax.scatter(range(n), res_sar, alpha=0.6, color='steelblue', s=40,
           label=f'SAR  (std={res_sar.std():.2f})')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel("Region index")
ax.set_ylabel("Residual")
ax.set_title("OLS vs SAR Residuals")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# (3) 계수 비교 (OLS vs SAR)
ax = axes[1, 0]
x_pos = np.arange(len(labels))
width = 0.35
bars1 = ax.bar(x_pos - width/2, ols_betas[:3], width,
               label='OLS', color='tomato', alpha=0.8, edgecolor='white')
bars2 = ax.bar(x_pos + width/2, sar_betas[:3], width,
               label='SAR (MLE)', color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel("Coefficient value")
ax.set_title("Coefficient Comparison: OLS vs SAR")
ax.axhline(0, color='black', linewidth=0.8)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.5 if bar.get_height() >= 0 else -2.5),
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.5 if bar.get_height() >= 0 else -2.5),
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# (4) 공간 시차 변수 (W·y) vs CRIME
ax = axes[1, 1]
wy = lag_spatial(w, y_flat)
ax.scatter(wy, y_flat, alpha=0.7, color='mediumseagreen',
           edgecolors='white', s=60)
z  = np.polyfit(wy, y_flat, 1)
p  = np.poly1d(z)
xline = np.linspace(wy.min(), wy.max(), 100)
ax.plot(xline, p(xline), 'r--', linewidth=1.5,
        label=f'slope = {z[0]:.2f}')
ax.set_xlabel("Spatial Lag W·CRIME (neighbor avg)")
ax.set_ylabel("CRIME")
ax.set_title("Spatial Lag vs CRIME  (Moran's I intuition)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
output_path = "sar_libpysal_result.png"
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  시각화 저장 완료: {output_path}")


# =============================================================
# STEP 8. 최종 요약
# =============================================================
print("\n" + "=" * 60)
print("STEP 8. 최종 요약")
print("=" * 60)
print(f"""
  SAR 모델: CRIME = ρ·W·CRIME + β0 + β1·INC + β2·HOVAL + ε

  ┌──────────────────────────────────────────────────────┐
  │  ρ (공간 자기회귀 계수) = {rho_hat:.4f}                 │
  │  → 이웃 지역 범죄율의 양의 공간 전이 확인             │
  │                                                      │
  │  β_INC   = {sar_betas[1]:.4f}  (소득 ↑ → 범죄율 ↓)         │
  │  β_HOVAL = {sar_betas[2]:.4f}  (주택가치 ↑ → 범죄율 ↓)     │
  │                                                      │
  │  Moran's I  = {mi.I:.4f}  (p = {mi.p_norm:.4f})            │
  │  OLS  R²    = {ols.r2:.4f}                               │
  │  SAR  R²    = {sar.pr2:.4f}  (공간 구조 반영 후 향상)   │
  └──────────────────────────────────────────────────────┘

  핵심 포인트:
  1. Moran's I > 0 → 공간 자기상관 존재 → OLS 편향 → SAR 필요
  2. ρ > 0 : 범죄는 이웃 지역으로 전파되는 공간 클러스터링
  3. SAR pseudo R²가 OLS R²보다 높음 → 설명력 향상
  4. INC 계수 음수 → 소득 높은 지역일수록 범죄율 낮음
""")


STEP 2. 공간 가중치 행렬 W 생성 (Queen Contiguity)
  W 행렬 크기      : 49 x 49
  평균 이웃 수     : 4.82
  변환 방식        : Row-standardized (행 합 = 1)

STEP 3. Moran's I — 공간 자기상관 검정
  Moran's I    : 0.5002
  기대값 E[I]  : -0.0208
  p-value      : 0.0000
  결론         : 유의미한 양의 공간 자기상관 존재 → SAR 적합 ✓

STEP 4. OLS 모델 추정 (비교 기준)
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :    Columbus
Weights matrix      :     unknown
Dependent Variable  :       CRIME                Number of Observations:          49
Mean dependent var  :     35.1288                Number of Variables   :           3
S.D. dependent var  :     16.7321                Degrees of Freedom    :          46
R-squared           :      0.5524
Adjusted R-squared  :      0.5329
Sum squared residual:     6014.89                F-statistic           :     28.3856
Sigma-square        :     130.759                Prob(F-statistic)     :   9.341e-09
S.E. of r

# SEM

In [ ]:
import libpysal
import numpy as np
from spreg import OLS, ML_Lag, ML_Error

In [7]:
# 데이터 로딩
db  = libpysal.io.open(libpysal.examples.get_path('columbus.dbf'))
y   = np.array(db.by_col('CRIME')).reshape(-1, 1)
X   = np.array([db.by_col('INC'), db.by_col('HOVAL')]).T

w   = libpysal.io.open(libpysal.examples.get_path('columbus.gal')).read()
w.transform = 'r'

# ── STEP 1. OLS + 진단 통계로 SAR/SEM 판단 ──────────────
ols = OLS(y, X, w=w, name_y='CRIME', name_x=['INC','HOVAL'],
          spat_diag=True)
print(ols.summary)
# summary 하단 LM-Lag / LM-Error / RLM-Lag / RLM-Error 확인

# ── STEP 2. SEM 추정 (ML_Error) ──────────────────────────
sem = ML_Error(y, X, w=w, name_y='CRIME', name_x=['INC','HOVAL'],
               name_ds='Columbus')
print(sem.summary)

# ── STEP 3. 핵심 결과 비교 ───────────────────────────────
print(f"OLS  R²         : {ols.r2:.4f}")
print(f"SAR  pseudo R²  : {sar.pr2:.4f}")   # 앞서 추정한 SAR
print(f"SEM  pseudo R²  : {sem.pr2:.4f}")

print(f"\nSEM lambda (λ)  : {sem.lam:.4f}")  # 공간 오차 자기상관 계수
print(f"SEM β_INC       : {sem.betas[1][0]:.4f}")
print(f"SEM β_HOVAL     : {sem.betas[2][0]:.4f}")

# ```

# ---

# ### 결과 해석 포인트

# **λ (lambda) 해석:**
# - `λ > 0` : 이웃 지역의 누락변수/측정오차가 같은 방향으로 영향
# - `λ = 0` : 오차 간 공간 의존성 없음 → 일반 OLS와 동일
# - λ는 β 계수에 직접 영향을 주지 않고, **표준오차 추정을 교정**하는 역할

# **β 계수 비교 (Columbus 기준 예상값):**

# | 파라미터 | OLS | SAR | SEM |
# |--------|-----|-----|-----|
# | const | 61.23 | 53.52 | 59.89 |
# | INC | -2.16 | -2.01 | -2.08 |
# | HOVAL | -0.01 | -0.02 | -0.03 |
# | ρ / λ | — | 0.2610 | ~0.32 |

# ---

# ### SAR / SEM / OLS 모델 선택 흐름

# 데이터 준비
#     ↓
# OLS 추정 (spat_diag=True)
#     ↓
# Moran's I 확인
#     ├─ 유의하지 않음 → OLS 사용
#     └─ 유의함
#          ↓
#     LM-Lag vs LM-Error
#          ├─ LM-Lag만 유의    → SAR (ML_Lag)
#          ├─ LM-Error만 유의  → SEM (ML_Error)
#          └─ 둘 다 유의       → RLM 값 더 큰 쪽 선택
# ```

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
-----------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :       CRIME                Number of Observations:          49
Mean dependent var  :     35.1288                Number of Variables   :           3
S.D. dependent var  :     16.7321                Degrees of Freedom    :          46
R-squared           :      0.5524
Adjusted R-squared  :      0.5329
Sum squared residual:     6014.89                F-statistic           :     28.3856
Sigma-square        :     130.759                Prob(F-statistic)     :   9.341e-09
S.E. of regression  :      11.435                Log likelihood        :    -187.377
Sigma-square ML     :     122.753                Akaike info criterion :     380.754
S.E of regression ML:     11.0794                Schwarz criterion     :     386.430

------------------------------------------------------------

/Users/erickim/Documents/ML_Training/.venv/lib/python3.11/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
---------------------------------------------------
Data set            :    Columbus
Weights matrix      :     unknown
Dependent Variable  :       CRIME                Number of Observations:          49
Mean dependent var  :     35.1288                Number of Variables   :           3
S.D. dependent var  :     16.7321                Degrees of Freedom    :          46
Pseudo R-squared    :      0.5362
Log likelihood      :   -183.7494
Sigma-square ML     :     97.6742                Akaike info criterion :     373.499
S.E of regression   :      9.8830                Schwarz criterion     :     379.174

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        60.27947    

# SARMA